In [1]:
from google.colab import drive

from google import genai
from pydantic import BaseModel, Field
from typing import List, Literal, get_args
from google.colab import userdata

import regex
import json


import pickle

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Szakdoga/Codes
from literature_objects import Segment
%cd /content/drive/MyDrive/Szakdoga/Data

[Errno 2] No such file or directory: '/content/drive/MyDrive/Szakdoga/Codes'
/content


ModuleNotFoundError: No module named 'literature_objects'

## Get Chapters

In [ ]:
# Put the chapter titles/numbers in a list to chunk the book
import inflect
p = inflect.engine()
#p.number_to_words(i).upper()

starting_phrases = [f"Chapter {i}" for i in range(1, 18)]

ending_phrases = [f"Chapter {i}" for i in range(2, 18)] + ["And together they walked back through the gateway to the Muggle world."]



In [ ]:
name_of_text_file = "NAME.txt"

c = 0
test_data: list[Segment] = []
recording: bool = False

with open(name_of_text_file, "r") as f:
    for line in f:
        if starting_phrases[c] in line and not recording:
            test_data.append(scene := Segment(start_phrase=starting_phrases[c], end_phrase=ending_phrases[c],description=starting_phrases[c]))
            scene.text = line
            recording = True

        if ending_phrases[c] in line and recording:
            c += 1
            recording = False
            if c >= len(starting_phrases):
                break

            if starting_phrases[c] in line:
                test_data.append(scene := Segment(start_phrase=starting_phrases[c], end_phrase=ending_phrases[c],))
                scene.text = line
                recording = True

        elif recording:
            scene.text += line


## Jelenet darabolás
A gondolat az, hogy ideális lenne nem kézzel jelenetekre bontani a műveket, mert az nagyon sok munka, de szükséges lehet, mert úgy gondolom, hogy jóval hatékonyabb chunkonként feldolgozni egy művet, mint egyben.
A jelenet egy ideális méretű egységnek tűnik, mert tartalmi egységet képez, ám számban jóval alatta van a bekezdéseknek, pláne a mondatoknak.

In [ ]:
def fuzzy_split(text, delimiters, max_errors=1):
    # Create a pattern: (delim1|delim2|delim3){e<=1}
    # {e<=1} allows up to 1 error (insertion, deletion, or substitution)
    pattern = f"({'|'.join(delimiters)}){{e<={max_errors}}}"

    # Use regex.split to break the text
    parts = regex.split(pattern, text)
    joined_results = [parts[0]]
    for i in range(1, len(parts) - 2, 2):
        joined_results.append(parts[i] + " " + parts[i+1])

    joined_results[-1] += parts[-1]
    return joined_results

In [ ]:
#Segment the text for testing. I chose the second and third chapters.
start_phrase = "CHAPTER ONE"
end_phrase = "THE END"

In [ ]:
text = ""
recording = False


with open("HP1.txt", "r") as f:
    for line in f:
        if start_phrase in line:
            recording = True

        if end_phrase in line:
            recording = False
            break

        if recording:
            text += line.lower()


In [ ]:
## Set up the gemini api
model_name = "gemini-3-flash-preview"

## Client authentication
api_key = None
try:
    with open("api_auth.txt", "r") as f:
        api_key = f.readline().strip()
    print("API key loaded from api_auth.txt.")
except FileNotFoundError:
    print("No api_key file")

client = genai.Client(api_key=api_key)

#----------------------------------------------

# Generating resposne format
general_prompt = "In the following text, separate the different scenes from each other. One scene is logically coherent, usually at the same place in time and space, with mostly the same characters"
enlarge_scenes = ""
enlarge_scenes += "Two scenes are treated the same if one scene is only a story within the given scene" #in new line, so you can comment out variable's value without causing eeror in the api call

class Scene(BaseModel):
    start_phrase: str = Field(description="The first sentence of the scene.")
    end_phrase: str = Field(description="The last sentence of the scene.")
    description: str = Field(description="A 3 word description of the scene")

class SceneList(BaseModel):
    scenes: List[Scene] = Field(description="List of the scenes in the story.")



#Generating response
response = client.models.generate_content(
    model=model_name,
    contents=f"{general_prompt + enlarge_scenes} \n {text}",
    config={
        "response_mime_type": "application/json",
        "response_json_schema": SceneList.model_json_schema(),
    },
)

response_dict = json.loads(response.text)["scenes"]

API key loaded from api_auth.txt.


In [ ]:
c = 0
test_list: list[Segment] = []
recording: bool = False

for c, scene_text in enumerate(fuzzy_split(text, [s["start_phrase"] for s in response_dict][1:], 4)):
    print(c, end= "   ")
    test_list.append(scene := Segment(start_phrase=response_dict[c]["start_phrase"], end_phrase=response_dict[c]["end_phrase"], description=response_dict[c]["description"]))
    scene.text = scene_text

0   1   2   3   4   5   6   7   8   9   10   11   12   13   14   15   16   17   18   19   20   21   22   23   24   25   26   27   28   29   30   31   32   33   34   35   36   37   38   39   40   41   42   43   44   45   46   47   48   

In [ ]:
if len(test_list) != len(response_dict):
    print(f"extracted_scenes: {len(test_list)}")
    print(f"response_scenes: {len(response_dict)}")
    raise Exception("The number of scenes does not match the number of scenes in the response. \n Check the last segment correctly appended. \n Possibly issue: LLM generated wrong quote or the generated quote appears multiple times in the text.")

extracted_scenes: 49
response_scenes: 53


Exception: The number of scenes does not match the number of scenes in the response. 
 Check the last segment correctly appended. 
 Possibly issue: LLM generated wrong quote or the generated quote appears multiple times in the text.

## Get model response

In [ ]:
#Model parameters
model_name = "gemini-3-flash-preview"

prompt_file_name = "test_prompt.txt"

try:
    with open(prompt_file_name) as f:
        prompt = f.read()
        print(f"Prompt loaded from {prompt_file_name}")
except FileNotFoundError:
    print("No prompt file")

#---------------------------------------------------------

is_directed: bool = True

if is_directed:
    prompt += "Assimetries are allowed in an interaction scores."

sentiments = Literal[-2, -1, 0, 1, 2]

#Response formating

class Connection(BaseModel):
    name: str = Field(description="Name of the connection")
    description: sentiments = Field(description=f"Description of the connection with {max(get_args(sentiments))} being very positive, 0 neutral, and {min(get_args(sentiments))} very negative")
    quote: str = Field(description="A characteristic quote from or about the interaction of the two characters") # in an effort to reduce pollution from training knowledge.
    # weight: int = Field(description="An int between 1 and 100 to describe how important this connection is likely to be")

class Character(BaseModel):
    name: str = Field(description="Name of the character")
    connections: List[Connection] = Field(description="List of the names of the connections of the character")

class CharacterList(BaseModel):
    characters: List[Character] = Field(description="List of the characters in the story")

## Client authentication
api_key = None
try:
    with open("api_auth.txt", "r") as f:
        api_key = f.readline().strip()
    print("API key loaded from api_auth.txt.")
except FileNotFoundError:
    print("No api_key file")

client = genai.Client(api_key=api_key)

Prompt loaded from test_prompt.txt
API key loaded from api_auth.txt.


In [ ]:
responses: list[dict] = []

#test_data: list[Segment] = test_list #change this dict to change the test set

for i, scene in enumerate(test_data):
    print(f"Generating response for scene No. {i+1}/{len(test_data)}", flush=True)

    response = client.models.generate_content(
        model=model_name,
        contents=f"{prompt} \n {scene.text}",
        config={
            "response_mime_type": "application/json",
            "response_json_schema": CharacterList.model_json_schema(),
        },
    )
    scene.network = json.loads(response.text)
    responses.append(scene.network)


Generating response for scene No. 1/17
Generating response for scene No. 2/17
Generating response for scene No. 3/17
Generating response for scene No. 4/17
Generating response for scene No. 5/17
Generating response for scene No. 6/17
Generating response for scene No. 7/17
Generating response for scene No. 8/17
Generating response for scene No. 9/17
Generating response for scene No. 10/17
Generating response for scene No. 11/17
Generating response for scene No. 12/17
Generating response for scene No. 13/17
Generating response for scene No. 14/17
Generating response for scene No. 15/17
Generating response for scene No. 16/17
Generating response for scene No. 17/17


In [ ]:
for i in responses[0]['characters']:
    print(i)

{'name': 'Harry Potter', 'connections': [{'name': 'Vernon Dursley', 'description': -2, 'quote': 'WHAT HAVE I TOLD YOU," thundered his uncle, spraying spit over the table, "ABOUT SAYING THE ‘M’ WORD IN OUR HOUSE?'}, {'name': 'Petunia Dursley', 'description': -2, 'quote': 'Harry knew he shouldn’t have risen to Dudley’s bait... Aunt Petunia knew he hadn’t really done magic, but he still had to duck as she aimed a heavy blow at his head with the soapy frying pan.'}, {'name': 'Dudley Dursley', 'description': -2, 'quote': 'Today’s your birthday," sneered Dudley. "How come you haven’t got any cards? Haven’t you even got friends at that freak place?'}, {'name': 'Hedwig', 'description': 1, 'quote': 'She’s bored," he said. "She’s used to flying around outside. If I could just let her out at night —'}, {'name': 'Ron Weasley', 'description': -1, 'quote': 'The long silence from Ron and Hermione had made Harry feel so cut off from the magical world that even taunting Dudley had lost its appeal.'}, {

## Regularizing names

In [ ]:
names: set = set()

for response in responses:
    for character in response['characters']:
        names.add(character['name'])

        for connection in character['connections']:
            names.add(connection['name'])

In [ ]:
class Names(BaseModel):
    name: str = Field(description="Full name of the character")
    aliases: List[str] = Field(description="List of the aliases of the character. Include the name in the name filed")

class NamesList(BaseModel):
    names: List[Names] = Field(description="List of the names of the characters in the story")

response = client.models.generate_content(
    model=model_name,
    contents=f"In the following list, cluster together the different names for the characters: \n {list(names)}",
     config={
            "response_mime_type": "application/json",
            "response_json_schema": NamesList.model_json_schema(),
        },
    )

name_dict = json.loads(response.text)['names']
name_map = { alias : name['name'] for name in name_dict for alias in name['aliases']}


In [ ]:
for scene in test_data:
    for character in scene.network['characters']:
        character["name"] = name_map[character["name"]]

        for connection in character['connections']:
            connection['name'] = name_map[connection['name']]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Saving

In [ ]:
pickle.dump(test_data, open("NAME_by_chapter.p", "wb"))